In [1]:
# ==========================================
# BLOCK 1: DATA INGESTION & CLEANING
# ==========================================
import pandas as pd
import numpy as np
import os

print("Loading UCI E-Commerce Dataset...")
data_path = "../data"
# Note: UCI dataset often contains special characters, so we use unicode_escape
df = pd.read_csv(os.path.join(data_path, "data.csv"), encoding='unicode_escape')

print(f"Initial dataset shape: {df.shape}")

# 1. Drop Missing Customer IDs
# We cannot perform RFM clustering on anonymous transactions
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Filter out Returns and Invalid Prices
# Since we are doing pure RFM, we only want positive, successful revenue-generating orders
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 3. Format the Date Column
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 4. Calculate the Total Amount per line item
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 5. Clean up the CustomerID format (Pandas loads it as a float by default)
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Total unique customers: {df_clean['CustomerID'].nunique()}")
display(df_clean.head())

Loading UCI E-Commerce Dataset...
Initial dataset shape: (541909, 8)
Cleaned dataset shape: (397884, 9)
Total unique customers: 4338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [2]:
# ==========================================
# BLOCK 2: PURE RFM AGGREGATION & CURRENCY CONVERSION
# ==========================================

# 1. CURRENCY CONVERSION (GBP to INR)
# We apply a ~105 multiplier so the model trains natively on the INR scale
GBP_TO_INR = 105
df_clean['TotalAmount'] = df_clean['TotalAmount'] * GBP_TO_INR

# 2. Define the "Snapshot Date" (The "current" day)
# We take the latest date in the dataset and add 1 day so the most recent buyer has Recency = 1
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# 3. Aggregate R, F, and M for each unique customer
df_rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # RECENCY: Days since last order
    'InvoiceNo': 'nunique',                                   # FREQUENCY: Count of unique orders
    'TotalAmount': 'sum'                                      # MONETARY: Total money spent (in INR)
}).reset_index()

# 4. Rename columns to our standard ML pipeline names
df_rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

# Set CustomerID back as the index for the ML model
df_rfm.set_index('CustomerID', inplace=True)

print(f"RFM Dataset shape: {df_rfm.shape}")
print("\n--- RFM Summary Statistics (INR Scale) ---")
display(df_rfm.describe().round(2))
display(df_rfm.head())

RFM Dataset shape: (4338, 3)

--- RFM Summary Statistics (INR Scale) ---


,Recency,Frequency,Monetary
count,4338.00,4338.00,4338.00
mean,92.54,4.27,215697.98
std,100.01,7.70,943869.20
min,1.00,1.00,393.75
25%,18.00,1.00,32278.58
50%,51.00,2.00,70820.92
75%,142.00,5.00,174482.70
max,374.00,209.00,29421632.10


,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,8104278.00
12347,2,7,452550.00
12348,75,4,188710.20
12349,19,1,184542.75
12350,310,1,35112.00


In [3]:
# ==========================================
# CHECK MONETARY SKEWNESS
# ==========================================

from scipy.stats import skew

print("Monetary skewness:", skew(df_rfm["Monetary"]))

print("\nMonetary percentiles:")
print(df_rfm["Monetary"].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00]))

Monetary skewness: 19.318270384225368

Monetary percentiles:
0.50    7.082093e+04
0.75    1.744827e+05
0.90    3.828858e+05
0.95    6.133935e+05
0.99    2.087505e+06
1.00    2.942163e+07
Name: Monetary, dtype: float64


In [4]:
# ==========================================
# TEST LOG TRANSFORMATION
# ==========================================

df_rfm["Monetary_Log"] = np.log1p(df_rfm["Monetary"])

print("Original Monetary skewness:",
      skew(df_rfm["Monetary"]))

print("Log-transformed Monetary skewness:",
      skew(df_rfm["Monetary_Log"]))

df_rfm["Frequency_Log"] = np.log1p(df_rfm["Frequency"])

print("Original Frequency skewness:",
      skew(df_rfm["Frequency"]))

print("Log-transformed Frequency skewness:",
      skew(df_rfm["Frequency_Log"]))

print("Recency skewness:",
      skew(df_rfm["Recency"]))

Original Monetary skewness: 19.318270384225368
Log-transformed Monetary skewness: 0.3809319352979806
Original Frequency skewness: 12.062857869870964
Log-transformed Frequency skewness: 1.2082335351584435
Recency skewness: 1.2456166142880103


In [5]:
# ==========================================
# BLOCK 3: OUTLIER REMOVAL, SCALING & ELBOW METHOD
# ==========================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import joblib
import os

df_math = df_rfm[["Recency", "Frequency", "Monetary"]].copy()

# Log-transform only the skewed variables
df_math["Frequency"] = np.log1p(df_math["Frequency"])
df_math["Monetary"] = np.log1p(df_math["Monetary"])

print(df_math.describe().round(2))

# ==========================================
# STANDARDIZE THE TRANSFORMED FEATURES
# ==========================================

scaler = StandardScaler()

scaled_data = scaler.fit_transform(df_math)

df_scaled = pd.DataFrame(
    scaled_data,
    columns=df_math.columns,
    index=df_math.index
)

# Save the scaler using the SAME filename as before
scaler_path = "../models/scaler.pkl"
joblib.dump(scaler, scaler_path)

print(f"Scaler saved to: {scaler_path}")

print("\nScaled feature statistics:")
print(df_scaled.describe().round(2))

# ==========================================
# FIND THE BEST K
# ==========================================

from sklearn.metrics import silhouette_score

inertia_values = []
silhouette_values = []

k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(df_scaled)

    inertia_values.append(kmeans.inertia_)
    silhouette_values.append(silhouette_score(df_scaled, labels))

for k, inertia, silhouette in zip(
    k_range, inertia_values, silhouette_values
):
    print(
        f"K={k} | "
        f"Inertia={inertia:.2f} | "
        f"Silhouette={silhouette:.4f}"
    )




       Recency  Frequency  Monetary
count  4338.00    4338.00   4338.00
mean     92.54       1.35     11.24
std     100.01       0.68      1.26
min       1.00       0.69      5.98
25%      18.00       0.69     10.38
50%      51.00       1.10     11.17
75%     142.00       1.79     12.07
max     374.00       5.35     17.20
Scaler saved to: ../models/scaler.pkl

Scaled feature statistics:
       Recency  Frequency  Monetary
count  4338.00    4338.00   4338.00
mean      0.00      -0.00      0.00
std       1.00       1.00      1.00
min      -0.92      -0.96     -4.18
25%      -0.75      -0.96     -0.68
50%      -0.42      -0.36     -0.06
75%       0.49       0.65      0.65
max       2.81       5.86      4.72
K=2 | Inertia=6875.70 | Silhouette=0.4062
K=3 | Inertia=4302.61 | Silhouette=0.4155
K=4 | Inertia=3246.70 | Silhouette=0.3792
K=5 | Inertia=2765.35 | Silhouette=0.3431
K=6 | Inertia=2400.76 | Silhouette=0.3317
K=7 | Inertia=2169.84 | Silhouette=0.2991
K=8 | Inertia=1964.49 | Silhouette

In [6]:
# ==========================================
# FINAL K-MEANS MODEL & CLUSTER PROFILING
# ==========================================

print("Training final K-Means model with K=4...")

# Train using the corrected transformed + scaled RFM features
final_kmeans = KMeans(
    n_clusters=4,
    init="k-means++",
    random_state=42,
    n_init=10
)

final_kmeans.fit(df_scaled)

# Predict clusters for the same customers used during training
df_rfm_final = df_rfm[["Recency", "Frequency", "Monetary"]].copy()
df_rfm_final["Cluster"] = final_kmeans.predict(df_scaled)

# Save the final model using the SAME filename as before
model_path = "../models/kmeans_model.pkl"
joblib.dump(final_kmeans, model_path)

# Profile clusters using ORIGINAL business-friendly RFM values
cluster_profile = df_rfm_final.groupby("Cluster").agg({
    "Recency": "mean",
    "Frequency": "mean",
    "Monetary": "mean"
})

cluster_profile["Total_Customers"] = (
    df_rfm_final.groupby("Cluster").size()
)

cluster_profile = cluster_profile.sort_values(
    by="Recency"
)

print(f"Model saved to: {model_path}")
print("\n--- FINAL K=4 CLUSTER PROFILE ---")
display(cluster_profile.round(2))

# ==========================================
# VERIFY SAVED MODEL ASSETS
# ==========================================

saved_scaler = joblib.load("../models/scaler.pkl")
saved_model = joblib.load("../models/kmeans_model.pkl")

print("Scaler features:", saved_scaler.feature_names_in_)
print("Number of clusters:", saved_model.n_clusters)
print("Scaler and model loaded successfully.")

Training final K-Means model with K=4...
Model saved to: ../models/kmeans_model.pkl

--- FINAL K=4 CLUSTER PROFILE ---


,Recency,Frequency,Monetary,Total_Customers
Cluster,,,,
1,19.75,15.76,1024946.99,577
3,46.23,4.19,173074.86,1448
0,58.01,1.51,40430.32,1375
2,259.41,1.37,40617.63,938


Scaler features: ['Recency' 'Frequency' 'Monetary']
Number of clusters: 4
Scaler and model loaded successfully.


In [7]:
# ==========================================
# INTELLIGENT K=4 SEGMENTATION TEST
# ==========================================

test_cases = pd.DataFrame([
    # Expected High Value
    {"Name": "High Value", "Recency": 15, "Frequency": 15, "Monetary": 1000000},

    # Expected Promising
    {"Name": "Promising", "Recency": 45, "Frequency": 4, "Monetary": 175000},

    # Expected At Risk
    {"Name": "At Risk", "Recency": 60, "Frequency": 2, "Monetary": 40000},

    # Expected Churned/Lost
    {"Name": "Churned/Lost", "Recency": 250, "Frequency": 1, "Monetary": 40000},

    # Recency boundary tests
    {"Name": "Risk Test 100d", "Recency": 100, "Frequency": 2, "Monetary": 40000},
    {"Name": "Risk Test 150d", "Recency": 150, "Frequency": 2, "Monetary": 40000},
    {"Name": "Risk Test 200d", "Recency": 200, "Frequency": 2, "Monetary": 40000},
])

# Copy only the actual model features
test_features = test_cases[["Recency", "Frequency", "Monetary"]].copy()

# Apply the SAME transformations used during training
test_features["Frequency"] = np.log1p(test_features["Frequency"])
test_features["Monetary"] = np.log1p(test_features["Monetary"])

# Apply the SAME scaler
test_scaled = scaler.transform(test_features)

# Predict
test_cases["Predicted_Cluster"] = final_kmeans.predict(test_scaled)

# Calculate distance to every centroid
distances = final_kmeans.transform(test_scaled)

for i in range(4):
    test_cases[f"Distance_to_{i}"] = distances[:, i]

display(test_cases.round(3))

c:\customer_segmentation\venv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\customer_segmentation\venv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(


,Name,Recency,Frequency,Monetary,Predicted_Cluster,Distance_to_0,Distance_to_1,Distance_to_2,Distance_to_3
0,High Value,15,15,1000000,1,3.890,0.500,4.684,2.334
1,Promising,45,4,175000,3,1.700,1.781,2.816,0.163
2,At Risk,60,2,40000,0,0.344,3.101,2.049,1.253
3,Churned/Lost,250,1,40000,2,1.947,4.215,0.333,2.630
4,Risk Test 100d,100,2,40000,0,0.541,3.177,1.662,1.356
5,Risk Test 150d,150,2,40000,0,0.981,3.339,1.191,1.621
6,Risk Test 200d,200,2,40000,2,1.460,3.564,0.757,1.978
